# Day 03 — Mosaic
### #30DayChartChallenge | April 2026

**How you die depends on where you live.** In low-income countries, half of all deaths come from infections. In high-income countries, cancer and heart disease dominate. A mosaic chart showing cause of death by World Bank income group.

**Data:**  
WHO Global Health Estimates 2021: Deaths by Cause, Age, Sex, by World Bank Income Group.  
Source: [WHO GHE](https://www.who.int/data/gho/data/themes/mortality-and-global-health-estimates/ghe-leading-causes-of-death)  
Downloaded: July 2024 release

**Author:** Sharfudeen Yasar Arafath

In [8]:
# — packages ------------------------------------------------------------------
#install.packages(c("devtools", "remotes"))                          # run once
#devtools::install_github("haleyjeppson/ggmosaic")     # run once

library(ggplot2)
library(dplyr)
library(showtext)
library(sysfonts)
library(ggmosaic)

In [9]:
# — fonts ---------------------------------------------------------------------

font_add_google("Outfit", "outfit")
font_add_google("Roboto Condensed", "roboto_condensed")
showtext_auto()
showtext_opts(dpi = 300)

In [10]:
# — read data -----------------------------------------------------------------

df <- read.csv("../../data/day_03/who_ghe_2021_deaths_by_cause_and_income.csv",
               stringsAsFactors = FALSE)

# factor levels in meaningful order
df$income_group <- factor(df$income_group,
  levels = c("Low income", "Lower-middle income",
             "Upper-middle income", "High income")
)

df$cause <- factor(df$cause,
  levels = c("Infectious & maternal", "Cardiovascular diseases",
             "Cancers", "Other NCDs", "Injuries")
)

df

income_group,cause,deaths,share_of_income_group
<fct>,<fct>,<int>,<dbl>
Low income,Cardiovascular diseases,852522,16.4
Low income,Cancers,343185,6.6
Low income,Other NCDs,778830,15.0
Low income,Infectious & maternal,2566979,49.5
Low income,Injuries,648448,12.5
Lower-middle income,Cardiovascular diseases,5462434,22.6
Lower-middle income,Cancers,2014274,8.3
Lower-middle income,Other NCDs,4697362,19.5
Lower-middle income,Infectious & maternal,9221016,38.2


In [11]:
# — build mosaic data ---------------------------------------------------------

# total deaths per income group (for column labels)
group_totals <- df %>%
  group_by(income_group) %>%
  summarise(group_deaths = sum(deaths), .groups = "drop")

cat("Deaths by income group (millions):\n")
group_totals %>% mutate(millions = round(group_deaths / 1e6, 1))

Deaths by income group (millions):


income_group,group_deaths,millions
<fct>,<int>,<dbl>
Low income,5189964,5.2
Lower-middle income,24139175,24.1
Upper-middle income,23844790,23.8
High income,14731202,14.7


In [12]:
# — theme & palette -----------------------------------------------------------

bg      <- "#0D1117"
txt     <- "#E6EDF3"
txt_dim <- "#8B949E"
txt_cap <- "#484F58"

pal <- c(
  "Infectious & maternal"    = "#E67E22",
  "Cardiovascular diseases"  = "#E63946",
  "Cancers"                  = "#9B59B6",
  "Other NCDs"               = "#3498DB",
  "Injuries"                 = "#1ABC9C"
)

In [ ]:
# — plot ----------------------------------------------------------------------

# build base mosaic to extract tile positions
p_base <- ggplot(df) +
  geom_mosaic(
    aes(weight = deaths, x = product(cause, income_group), fill = cause),
    offset = 0.012
  )

# extract tile coordinates for label placement
mosaic_data <- ggplot_build(p_base)$data[[1]]
mosaic_data$x_center <- (mosaic_data$xmin + mosaic_data$xmax) / 2
mosaic_data$y_center <- (mosaic_data$ymin + mosaic_data$ymax) / 2
mosaic_data$tile_h   <- mosaic_data$ymax - mosaic_data$ymin

# correct percentages: normalize tile heights within each column
# (this cancels out the offset/gap effect and avoids row-ordering issues)
mosaic_data <- mosaic_data %>%
  group_by(xmin) %>%
  mutate(pct = paste0(round(tile_h / sum(tile_h) * 100), "%")) %>%
  ungroup()

# text size: smaller for tiny tiles
mosaic_data$label_size <- ifelse(mosaic_data$tile_h >= 0.08, 4.5, 3.2)

# final plot
p <- ggplot(df) +
  geom_mosaic(
    aes(weight = deaths, x = product(cause, income_group), fill = cause),
    offset = 0.012, color = bg, linewidth = 0.8
  ) +

  # percentage labels inside tiles
  geom_text(
    data = mosaic_data,
    aes(x = x_center, y = y_center, label = pct, size = label_size),
    color = "white", family = "outfit", fontface = "bold",
    alpha = 0.9, inherit.aes = FALSE, show.legend = FALSE
  ) +
  scale_size_identity() +

  scale_fill_manual(values = pal, name = NULL) +

  labs(
    title    = "How You Die Depends on Where You Live",
    subtitle = paste0(
      "Cause of death by World Bank income group, 2021 ",
      "\u00b7 Column width = share of global deaths"
    ),
    x        = NULL,
    y        = NULL,
    caption  = paste0(
      "Data: WHO Global Health Estimates 2021 ",
      "\u2014 Deaths by Cause, Age, Sex, by World Bank Income Group.\n",
      "Source: who.int/data/gho/data/themes/mortality-and-global-health-estimates\n",
      "#30DayChartChallenge \u00b7 Day 03: Mosaic \u00b7 ",
      "Sharfudeen Yasar Arafath"
    )
  ) +

  theme_void(base_family = "roboto_condensed") +
  theme(
    plot.title       = element_text(family = "outfit", face = "bold", size = 32,
                                    hjust = 0, color = txt,
                                    margin = margin(t = 20, b = 8)),
    plot.subtitle    = element_text(size = 14, hjust = 0, color = txt_dim,
                                    margin = margin(b = 20)),
    plot.caption     = element_text(size = 10, hjust = 0, color = txt_cap,
                                    margin = margin(t = 20, b = 10),
                                    lineheight = 1.5),
    legend.position  = "top",
    legend.justification = "left",
    legend.text      = element_text(size = 14, color = txt_dim),
    legend.margin    = margin(b = 10),
    axis.text.x      = element_text(size = 14, color = txt_dim,
                                    margin = margin(t = 8)),
    plot.background  = element_rect(fill = bg, color = NA),
    panel.background = element_rect(fill = bg, color = NA),
    plot.margin      = margin(25, 30, 20, 20)
  )

p

In [14]:
# — save ----------------------------------------------------------------------

ggsave("../../chart/day_03_mosaic.png",
       plot = p, width = 13, height = 10, dpi = 300, bg = bg)

cat("Done — saved to chart/day_03_mosaic.png\n")

Done — saved to chart/day_03_mosaic.png
